#### Define a few parameters and functions

In [ ]:
import numpy as np, pandas as pd, scipy.special as ss, cmath, lmfit as lm
# set markersize for plotting later
markersize = 30
# set a standard water density and sediment density (kg/m^3)
rho, drho, rhos = 995, 10, 2650
# set constants: gravitational acceleration (m/s^2), kinematic viscosity (m^2/s), von Karman constant
g, nu, kappa = 9.81, 1e-6, 0.41
# Wong and Parker function and its inverse
wp_func = lambda x: 4.61*(x-0.051)**1.5
wp_inv_func = lambda x: (x/4.61)**(2/3) + 0.051
# wp_func = lambda x: 4*(x-0.0495)**1.5
# wp_inv_func = lambda x: (x/4)**(2/3) + 0.0495

# function to calculate the hydraulic radius correction factor
# inputs: alpha = W/H
def calc_FR(alpha): 
    return 1/(1 + 2/alpha)

# function to calculate the Vanoni-Brooks correction factor
# inputs: f = gRS/V^2, alpha = W/H, Re = VR/nu
def calc_FVB(f, alpha, Re, output='FVB'):
        # define constants
        a, b = 0.018409368, 0.224072094
        
        # calculate the ratio of wall friction to total friction (tau_wall/tau)
        twt = a * np.real(ss.lambertw(b * (Re/f)**(1/3)))**(-2) / f
        
        # output the correction factor
        if output == 'FVB':
            return 1 - twt / (1 + alpha/2)
        
        # output just the ratio of wall friction to total friction (tau_wall/tau)
        elif output == 'twt':
            return twt

# function to calculate the Guo correction factor
# inputs: f = gRS/V^2, alpha = W/H, ro = 30H/ks = 15H/D50
def calc_FG(f, alpha, ro, output='FG'):
        # define constants and important functions
        beta = (1/kappa) * (3/4 - 1/np.pi**2)
        phi = np.exp(-(alpha/np.pi)**(3/2))
        psi = np.sqrt(2/alpha) * (np.log(ro*np.exp(-1))/kappa - (1 + phi)**3/12/kappa + ((1 + np.sqrt(phi))/2)*(1+np.sin(np.pi*phi)/(np.pi*(1+phi))))
        
        # calculate the ratio of wall friction to total friction (tau_wall/tau)
        # two different ways to handle inputs depending on if they are arrays or not
        try:
            tcomplex = np.array([cmath.sqrt(xi) for xi in f*(1+alpha/2)*(psi**2 + beta**2) - 1])
        except:
            tcomplex = cmath.sqrt(f*(1+alpha/2)*(psi**2 + beta**2) - 1)
        
        twt = np.real((psi*tcomplex - beta)**2) / ((psi**2 + beta**2)**2 * f)
            
        if output == 'FG':         
            return 1 - twt/(1+alpha/2)
        
        elif output == 'twt':
            return twt

#### Load various data sources and homogenize them

##### Load Deal 2023

###### Calculate shape parameters

In [ ]:
grains = ['gb', 'ng']
filepath = 'flume_data/Deal_2023_grain_params/1_grain_properties/main_granular_materials'
df_density = pd.read_csv('{}/1_var_density.csv'.format(filepath)).set_index('Unnamed: 0').rename_axis(None)
df_shape = pd.read_csv('{}/2_var_shape.csv'.format(filepath)).set_index('Unnamed: 0').rename_axis(None)
df_ars = pd.read_csv('{}/3_var_ARS.csv'.format(filepath)).set_index('Unnamed: 0').rename_axis(None)
df_vels = pd.read_csv('{}/4_var_settling_velocities.csv'.format(filepath)).set_index('Unnamed: 0').rename_axis(None)
df_VES = pd.read_csv('{}/5_var_VES_settling_velocity.csv'.format(filepath)).set_index('Unnamed: 0').rename_axis(None)

# calculate cstar and mustar
bed_angle, aor_gb = 3.5, 24
cstar, dcstar, mustar, dmustar, ratio, dratio, Co, CDsettle, CD = {}, {}, {}, {}, {}, {}, {}, {}, {}
for grain in grains:
    cstar[grain] = ((df_VES.ws_sphere[grain])/(df_vels.vel_mean[grain]))**2 * df_shape.CSF[grain]
    dcstar[grain] = cstar[grain] * np.sqrt((2*df_vels.dvel_mean[grain]/df_vels.vel_mean[grain])**2 + (df_shape.dCSF[grain]/df_shape.CSF[grain])**2)

    mugb = (np.tan(np.radians(aor_gb)) - np.tan(np.radians(bed_angle)))
    mustar[grain] = (np.tan(np.radians(df_ars.mean_angle[grain])) - np.tan(np.radians(bed_angle)))/mugb
    dmustar[grain] = ((1 + np.tan(np.radians(df_ars.mean_angle[grain]))**2)/mugb) * np.radians(df_ars.mean_angle[grain]) * (df_ars.std_angle[grain]/df_ars.mean_angle[grain])
    ratio[grain] = cstar[grain]/mustar[grain]
    dratio[grain] = ratio[grain] * np.sqrt((dcstar[grain]/cstar[grain])**2 + (dmustar[grain]/mustar[grain])**2)
    
    Co[grain] = df_VES.CD_VES[grain]
    CDsettle[grain] = 4*(df_density.total_grain_ro[grain]/rho - 1)*g*df_shape.dn[grain] / (3*df_vels.vel_mean[grain]**2)
    CD[grain] = df_vels.CD[grain]
df_shape_params = pd.DataFrame({'Co': df_VES.CD_VES, 'CD': df_vels.CD, 'mustar': mustar, 'Cstar': cstar, 'dmustar': dmustar, 'dCstar': dcstar, 'Fshape': ratio, 'dFshape': dratio})

df_shape_params = df_shape_params.dropna(subset=['Fshape'])
df_vels

In [ ]:
df_in = pd.read_csv('flume_data/Deal_2023_grain_params/3_exp_meta_data/exp_data.txt')
flume_width = 0.0105 # m
dflume_width = 0.001 # m

df_in['FS'] = np.nan
df_in.loc[df_in.grain_kind.values == 'gb', 'FS'] = df_shape_params['Fshape']['gb']
df_in.loc[df_in.grain_kind.values == 'ng', 'FS'] = df_shape_params['Fshape']['ng']

# drop the rows of df_in where the values of df_in.tau8 = np.nan
df_in = df_in.dropna(subset=['tau8'])
df_in.loc[df_in.grain_kind.values == 'gb', 'grain_kind'] = 'sphere'
df_in.loc[df_in.grain_kind.values == 'ng', 'grain_kind'] = 'gravel'

# error analysis for shear stress
tau_norm = (df_in.grain_density.values - df_in.fluid_density.values)*g*df_in.VES_grain_diameter.values
dtau_norm = tau_norm * np.sqrt((df_in.dgrain_density.values/df_in.grain_density.values)**2 + (df_in.dfluid_density.values/df_in.fluid_density.values)**2 + (df_in.dVES_grain_diameter.values/df_in.VES_grain_diameter.values)**2)
taub = df_in.fluid_density.values * g * df_in.flow_depth.values * df_in.water_slope.values
dtaub = taub * np.sqrt((df_in.dfluid_density.values/df_in.fluid_density.values)**2 + (df_in.dflow_depth.values/df_in.flow_depth.values)**2 +(df_in.dwater_slope.values/df_in.water_slope.values)**2)
tau8 =  taub / tau_norm 
dtau8 = tau8 * np.sqrt((dtaub/taub)**2 + (dtau_norm/tau_norm)**2)

# error analysis for sediment flux
q8 = df_in.q8.values
dq8 = df_in.dq8.values

df_data_2 = pd.DataFrame({'W': flume_width * np.ones(len(df_in)),
                          'H': df_in.flow_depth.values, 
                          'S': df_in.water_slope.values, 
                          'Q': df_in.discharge.values/1000, 
                          'D': df_in.VES_grain_diameter.values, 
                          'rf': df_in.grain_density.values/df_in.fluid_density.values - 1,
                          'q8': df_in.q8.values,
                          'dq8': df_in.dq8.values,
                          'tau8': tau8, 
                          'dtau8': dtau8,
                          'FS': df_in.FS.values, 
                          'grain_kind': df_in.grain_kind.values, 
                          'flume_width': ['narrow' for i in range(df_in.grain_kind.values.size)], 
                          'data_source': 'Deal et al., 2023', 
                          'data': 2, 
                          'm': 's', 
                          'markersize': markersize-10})

df_data_2 = df_data_2.dropna(subset=['S'])

#####  X-mas, 2023 exp data

###### Calculate shape parameters

In [ ]:
grains = ['gb', 'ng']
filepath = 'flume_data/wide_flume_exp_data/material_props/'
df_density = pd.read_csv('{}/1_var_density.csv'.format(filepath)).set_index('Unnamed: 0').rename_axis(None)
df_shape = pd.read_csv('{}/2_var_shape.csv'.format(filepath)).set_index('Unnamed: 0').rename_axis(None)
df_ars = pd.read_csv('{}/3_var_ARS.csv'.format(filepath)).set_index('Unnamed: 0').rename_axis(None)
df_vels = pd.read_csv('{}/4_var_settling_velocities.csv'.format(filepath)).set_index('Unnamed: 0').rename_axis(None)
df_VES = pd.read_csv('{}/5_var_VES_settling_velocity.csv'.format(filepath)).set_index('Unnamed: 0').rename_axis(None)

# calculate cstar and mustar
bed_angle, aor_gb = 1.26, 24
cstar, dcstar, mustar, dmustar, ratio, dratio, Co, CDsettle, CD = {}, {}, {}, {}, {}, {}, {}, {}, {}
for grain in grains:
    cstar[grain] = ((df_VES.ws_sphere[grain])/(df_vels.vel_mean[grain]))**2 * df_shape.CSF[grain]
    dcstar[grain] = cstar[grain] * np.sqrt((2*df_vels.dvel_mean[grain]/df_vels.vel_mean[grain])**2 + (df_shape.dCSF[grain]/df_shape.CSF[grain])**2)

    mugb = (np.tan(np.radians(aor_gb)) - np.tan(np.radians(bed_angle)))
    mustar[grain] = (np.tan(np.radians(df_ars.mean_angle[grain])) - np.tan(np.radians(bed_angle)))/mugb
    dmustar[grain] = ((1 + np.tan(np.radians(df_ars.mean_angle[grain]))**2)/mugb) * np.radians(df_ars.mean_angle[grain]) * (df_ars.std_angle[grain]/df_ars.mean_angle[grain])
    ratio[grain] = cstar[grain]/mustar[grain]
    dratio[grain] = ratio[grain] * np.sqrt((dcstar[grain]/cstar[grain])**2 + (dmustar[grain]/mustar[grain])**2)
    
    Co[grain] = df_VES.CD_VES[grain]
    CDsettle[grain] = 4*(df_density.total_grain_ro[grain]/rho - 1)*g*df_shape.dn[grain] / (3*df_vels.vel_mean[grain]**2)
    CD[grain] = df_vels.CD[grain]
df_shape_params = pd.DataFrame({'Co': df_VES.CD_VES, 'CD': df_vels.CD, 'mustar': mustar, 'Cstar': cstar, 'dmustar': dmustar, 'dCstar': dcstar, 'Fshape': ratio, 'dFshape': dratio})

df_shape_params = df_shape_params.dropna(subset=['Fshape'])
df_shape_params

In [ ]:
flume_width = 0.065 # m
dflume_width = 0.005 # m

df_in_gb = pd.read_csv('flume_data/wide_flume_exp_data/flume_experiment_data/gb_exps/gb_5_data.csv')
df_in_ng = pd.read_csv('flume_data/wide_flume_exp_data/flume_experiment_data/ng_exps/ng_5_data.csv')

dfts = {'gb': df_in_gb, 'ng': df_in_ng}
gk = {'gb': 'sphere', 'ng': 'gravel'}
for key in ['gb', 'ng']:
    dft = dfts[key]
    dft['grain_density'] = np.ones(len(dft)) * df_density['total_grain_ro'][key]
    dft['dgrain_density'] = np.ones(len(dft)) * df_density['dtotal_grain_ro'][key]
    dft['FS'] = np.ones(len(dft)) * df_shape_params['Fshape'][key]
    dft['VES_grain_diameter'] = np.ones(len(dft)) * df_shape['dn'][key]
    dft['dVES_grain_diameter'] = np.ones(len(dft)) * df_shape['ddn'][key]
    dft['grain_kind'] = gk[key]
    
df_in = pd.concat([dfts['gb'], dfts['ng']], axis=0)
df_in['discharge'] = np.ones(len(df_in)) * np.mean([3.07480994, 2.76582836, 2.68489553])

# error analysis for shear stress
# dflowdepth?
# dslope?
slope = np.tan(np.radians(df_in.water_slope.values))
tau_norm = (df_in.grain_density.values - rho)*g*df_in.VES_grain_diameter.values
dtau_norm = tau_norm * np.sqrt((df_in.dgrain_density.values/df_in.grain_density.values)**2 + (drho/rho)**2 + (df_in.dVES_grain_diameter.values/df_in.VES_grain_diameter.values)**2)
taub = rho * g * df_in.flow_depth.values * slope
# dtaub = taub * np.sqrt((drho/rho)**2 + (df_in.dflow_depth.values/df_in.flow_depth.values)**2 +(dslope/slope)**2)
tau8 =  taub / tau_norm 
# dtau8 = tau8 * np.sqrt((dtaub/taub)**2 + (dtau_norm/tau_norm)**2)

# error analysis for sediment flux
# dequil_mass_flux?
qs_norm = np.sqrt(g*df_in.VES_grain_diameter.values*(df_in.grain_density.values-rho)/df_in.grain_density.values)*df_in.VES_grain_diameter.values
dqs_norm = np.sqrt((g*df_in.VES_grain_diameter.values**3)/(4*df_in.grain_density.values*(df_in.grain_density.values-rho))) * np.sqrt( (3*(df_in.grain_density.values-rho)*df_in.dVES_grain_diameter.values/df_in.VES_grain_diameter.values)**2 + (rho*df_in.dgrain_density.values/df_in.grain_density.values)**2 + drho**2)

qs = df_in.mass_flux.values / (rho*flume_width) # m^2/s # volume flux per unit channel width
# dqs = qs * np.sqrt( (dequil_mass_flux/df_in.mass_flux.values)**2 + (drho/rho)**2 + (dflume_width/flume_width)**2 )
q8 = qs / qs_norm # nondimensional volume flux per channel width
# dq8 = q8 * np.sqrt( (dqs/qs)**2 + (dqs_norm/qs_norm)**2 )

df_data_2b = pd.DataFrame({'W': flume_width * np.ones(len(df_in)),
                           'H': df_in.flow_depth.values,
                           'S': np.tan(np.radians(df_in.water_slope.values)),
                           'Q': df_in.discharge.values/1000,
                           'D': df_in.VES_grain_diameter.values, 
                           'rf': df_in.grain_density.values/rho - 1, 
                           'q8': df_in.q8.values,
                        #    'dq8': df_in.q8.values*0.1,
                        #    'tau8': tau8, 
                        #    'dtau8': tau8*0.1,
                           'FS': df_in.FS.values, 
                           'grain_kind': df_in.grain_kind.values,
                           'flume_width': ['wide' for i in range(df_in.grain_kind.values.size)],
                           'data_source': 'New experiments', 
                           'data': 3, 
                           'm': '*',  
                           'markersize': markersize+5})

# make table 1
df_table_1 = pd.DataFrame({'Sediment kind': df_in.grain_kind.values,
              'Flume width (m)': flume_width * np.ones(len(df_in)),
              'Mean flow depth (m)': df_in.flow_depth.values,
              'Water surface slope (m/m)': np.tan(np.radians(df_in.water_slope.values)),
              'Fluid flux (m3/s)': df_in.discharge.values/1000,
              'Sediment mass flux (kg/s)': df_in.mass_flux.values})

df_table_2 = pd.DataFrame({'Sediment density (kg/m3)': df_density['total_grain_ro'].values,
                            'Long grain axis (m)': df_shape['A'].values,
                            'Middle grain axis (m)': df_shape['B'].values,
                            'Short grain axis (m)': df_shape['C'].values,
                            'VES diameter (m)': df_shape['dn'].values,
                            'Corey shape factor': df_shape['CSF'].values,
                            'Drag coefficient': df_vels['CD'].values,
                            'VES drag coefficient': df_VES['CD_VES'].values,
                            'Bulk coeff. of friction': np.tan(np.radians(df_ars['mean_angle'].values)),
                            'Flume slope': [np.tan(np.radians(1.26)), np.tan(np.radians(1.26))],
                            'Grain shape corr. factor': df_shape_params['Fshape'].values}, 
                            index=['spheres', 'gravel']).T

##### Load MPM and Brownlie data

In [ ]:
df_in = pd.read_csv('flume_data/Smart_1983_MPM_1948.csv')
df_data_1 = pd.DataFrame({'W': df_in.B.values, 
                          'H': df_in.H.values, 
                          'S': df_in.S.values, 
                          'Q': df_in.Q.values/1000, 
                          'D': df_in.Dm.values/1000,
                          'rf': df_in.GS.values-1, 
                          'q8': df_in.q8.values, 
                          'data_source': 'MPM, 1948', 
                          'data': 0, 
                          'm': '+', 
                          'markersize': markersize})

df_in = pd.read_csv('flume_data/Brownlie_1981_Gilbert_1914.csv')
df_data_1b = pd.DataFrame({'W': df_in.B.values, 
                           'H': df_in.H.values, 
                           'S': df_in.S.values, 
                           'Q': df_in.Q.values/1000, 
                           'D': df_in.Dm.values/1000, 
                           'rf': df_in.GS.values-1, 
                           'q8': df_in.q8.values, 
                           'data_source': 'Gilbert, 1914', 
                           'data': 1, 
                           'm': 'p', 
                           'markersize': markersize-10})

##### Load Williams data

In [ ]:
# load data
df_in = pd.read_csv('flume_data/Williams_1970.csv')
df_in['run'] = df_in.index.values + 1

# convert to SI units
conversion_factors = {
    'width': 0.3048,
    'unit_discharge': 0.3048**2,
    'slope': 1,
    'depth': 0.3048,
    'mean_velocity': 0.3048,
    'sed_trans_measure_period': 1,
    'unit_transport_rate': 0.45359237/0.3048,
    'mass_flux_in': 0.45359237,
    'surface_velocity': 0.3048,
    'water_temp': 1,
    'run': 1}
for key in df_in.columns: df_in[key] = df_in[key] * conversion_factors[key]

df_data_3 = pd.DataFrame({'W': df_in['width'].values,
                          'H': df_in['depth'].values,
                          'S': df_in['slope'].values,
                          'Q': df_in['unit_discharge'].values * df_in['width'].values,
                          'D': np.ones(len(df_in)) * 0.00135,
                          'rf': rhos/rho - 1,
                          'q8': (df_in['unit_transport_rate']/rhos) / np.sqrt((rhos/rho - 1) * g * 0.00135**3), 
                          'data_source': 'Williams, 1970', 
                          'data': 4, 
                          'm': '.', 
                          'markersize': markersize-20})

##### Load Song 1994 data

In [ ]:
df_in = pd.read_csv('flume_data/Guo_2015_Song_1994.csv')
mask = df_in.flow_type == 'uniform'

df_data_4 = pd.DataFrame({'W': (df_in['h'][mask].values/100) * df_in['ar'][mask].values, 
                          'H': df_in['h'][mask].values/100,
                          'S': df_in['So'][mask].values/100,
                          'Q': df_in['V'][mask].values/100 * (df_in['h'][mask].values/100)**2 * df_in['ar'][mask].values,
                           # an inferred value of D is found by finding the value of ks used by Guo 2015 for the Song 1994 data, and then using a value of D which will recreate that value of ks in the FG_calc function. The code to do this is below, and the value found for ks is 0.0165 m.
                          'D': 0.008253 * np.ones(len(df_in['h'][mask].values)),
                          'rf': rhos/rho - 1, 
                          'tau_m': (df_in['ubc'][mask].values/100)**2 * rho,
                          'q8': np.nan * np.ones(len(df_in['h'][mask].values)), 
                          'data_source': 'Song, 1994', 
                          'data': 5, 
                          'm': 'd',
                          'markersize': (markersize + 10)})

##### Combine data

In [ ]:
df = pd.concat([df_data_1, df_data_1b, df_data_2, df_data_2b, df_data_3, df_data_4], axis=0)
df['V'] = df.Q / (df.W * df.H) # flow velocity in m/s
df['R'] = df.W * df.H / (df.W + 2 * df.H) # hydraulic radius in m
df['Re'] = df.V * df.R / nu # Reynolds number (dimensionless)
df['Fr'] = df.V / np.sqrt(g * df.R) # Froude number (dimensionless)
df['f'] = df.R * g * df.S / df.V**2 # friction factor (dimensionless)
df['ro'] = 15*df.H/df.D # relative roughness (dimensionless)
df['alpha'] = df.W/df.H # aspect ratio (dimensionless)
df['FR'] = calc_FR(df.alpha) # Sidewall correction based on aspect ratio (dimensionless)
df['FVB'] = calc_FVB(df.f, df.alpha, df.Re) # Sidewall correction from Einstein and Johnson (dimensionless)
df['FG'] = calc_FG(df.f, df.alpha, df.ro) # Sidewall correction from Guo (dimensionless)
df['tau'] = rho * g * df.S * df.H # bed shear stress (Pa) based on depth-slope product
df['tau8'] = df.H * df.S / df.D / df.rf # dimensionless bed shear stress based on depth-slope product
mask = (df.data_source != 'Song, 1994').values
df.loc[mask, 'tau_m'] = wp_inv_func(df.q8.values[mask]) * rho * g * df.rf.values[mask] * df.D.values[mask] # bed shear stress (Pa) based on dimensionless sediment flux

df['Fqs'] = df.tau_m/df.tau # empirical correction factor 
df['twt'] = (1 - df.tau_m/df.tau) * (1 + df.alpha/2) # ratio of wall shear stress to total shear stress (dimensionless)
print(df.tau_m.values)



##### Load natural channel data

In [ ]:
df_river = pd.read_csv('river_data/HG_data_comp_simplified.csv').drop(columns=['Unnamed: 0'])
df_river['alpha'] = df_river['width']/df_river['depth']
cond = (df_river.bankfull == True) & ((df_river.river_class == 0) | (df_river.river_class == 1))
df_river = df_river[cond]
(df_river.alpha.values > 6).sum() / 1598

In [ ]:
import matplotlib.pyplot as plt

def fit_model_standard(tau8, q8):
    def func(x, c, xc): return c*(x - xc)**(3/2)

    x_data = tau8
    # dx_data = dtau8
    y_data = q8
    # dy_data = dq8

    model = lm.Model(func)
    model.set_param_hint('c', value=6, min=0, max=100)
    model.set_param_hint('xc', value=0.1, min=0, max=np.nanmin(x_data))
    params = model.make_params()
    # dx_plot = dx_data.copy()
    # dy_plot = dy_data.copy()
    # x_plot = x_data.copy()
    # y_plot = y_data.copy()
    results = model.fit(y_data[np.isfinite(x_data)], params, x=x_data[np.isfinite(x_data)])
    xc = results.result.params['xc'].value
    dxc = results.result.params['xc'].stderr
    c = results.result.params['c'].value
    dc = results.result.params['c'].stderr
    rmsd_standard = np.sqrt(np.mean((np.log10(y_data) - np.log10(func(x_data, c, xc)))**2))
    
    return func, xc, c, dxc, dc, rmsd_standard

df_wide = df[(df.alpha >= 8) & (df.data_source != 'Williams, 1970') & (df.data_source != 'Song, 1994')]

xo = df_wide.tau8.values
yo = df_wide.q8.values
x = xo[~np.isnan(xo) & ~np.isnan(yo)]
y = yo[~np.isnan(xo) & ~np.isnan(yo)]
func, xc, c, dxc, dc, rmsd_standard = fit_model_standard(x, y)

print('xc: {:.3f} +/- {:.3f}, c: {:.3f} +/- {:.3f}, rmsd: {:.3f}'.format(xc, dxc, c, dc, rmsd_standard))

c, xc = 4.8, 0.053
plt.loglog(x, y, '.', label='Data')
plt.plot(np.logspace(np.log10(xc), 0, 100), func(np.logspace(np.log10(xc), 0, 100), c, xc), 'r-', label=r'Fit: $c$ = {:.3f} $\pm$ {:.3f}, $x_c$ = {:.3f} $\pm$ {:.3f}'.format(c, dc, xc, dxc))

plt.ylim(.5e-3, 3)
df_wide
# plt.show()

#### Tables

##### Table 1

In [ ]:
# df_table_1.style
# print out table with set number of decimal places for each column
df_table_1.style.format({
    'Flume width (m)': '{:.3f}',
    'Mean flow depth (m)': '{:.3f}',
    'Water surface slope (m/m)': '{:.4f}',
    'Fluid flux (m3/s)': '{:.4f}',
    'Sediment mass flux (kg/s)': '{:.4f}'})

<!-- ##### Table 2 -->

In [ ]:
df_table_2.style.format({
    'Sediment density (kg/m3)': '{:.1f}',
    'Long grain axis (m)': '{:.4f}',
    'Middle grain axis (m)': '{:.4f}',
    'Short grain axis (m)': '{:.4f}',
    'VES diameter (m)': '{:.4f}',
    'Corey shape factor': '{:.2f}',
    'Drag coefficient': '{:.2f}',
    'VES drag coefficient': '{:.2f}',
    'Bulk coeff. of friction': '{:.2f}',
    'Flume slope': '{:.4f}',
    'Grain shape corr. factor': '{:.3f}'})

#### Figures

##### Import libraries for plotting

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format='retina'
from matplotlib.patches import Rectangle
from matplotlib.patches import FancyBboxPatch
import matplotlib.pyplot as plt, matplotlib as mpl, matplotlib.colors as mcolors, seaborn as sns, warnings, scipy.special as ss, lmfit as lm
from mpl_toolkits.axes_grid1.inset_locator import inset_axes
warnings.filterwarnings('ignore')
sns.set_context('paper')
params = {
    'text.latex.preamble': '\\usepackage{gensymb}',
    'image.origin': 'lower',
    'image.interpolation': 'nearest',
    'image.cmap': 'gray',
    'axes.grid': False,
    'savefig.dpi': 300,  # to adjust notebook inline plot size
    'axes.labelsize': 17, # fontsize for x and y labels (was 10)
    'axes.titlesize': 16,
    'font.size': 16, # was 10
    'legend.fontsize': 12, # was 10
    'xtick.labelsize': 16,
    'ytick.labelsize': 16,
    'text.usetex': True,
    'figure.figsize': [3.39, 2.10],
    # 'font.family': 'serif',
}
mpl.rcParams.update(params)
plt.style.use('tableau-colorblind10')
figwidth = 12
global_cmap = 'viridis'
# global_cmap = 'cividis'

##### Figure 1

In [ ]:
fig = plt.figure(figsize=(figwidth, figwidth/3))

x = 'alpha'
stat = 'frequency'
kde = True
log_scale = True
binrange = (-1.25, 3)
binwidth = 0.05
alpha = 0.4
color = {'river': '#0072B2', 'flume': '#E69F00'}

sns.histplot(df, color=color['flume'], x=x, kde=kde, alpha=alpha, log_scale=log_scale, binrange=binrange, binwidth=binwidth, stat=stat)
sns.histplot(df_river, color=color['river'], x=x, kde=kde, alpha=alpha, log_scale=log_scale, binrange=binrange, binwidth=binwidth, stat=stat)

plt.plot([1, 1], [0, 2000], 'k--')
plt.annotate('Very narrow flumes', (0.17, 1750), fontsize=12)
plt.arrow(0.9, 1500, -0.5, 0, head_width=125, head_length=0.05, fc='k', ec='k', lw=.5)

plt.plot([10, 10], [0, 3000], 'k--')
plt.annotate('Channel geometry\nmodifies bed stress', (1.8, 2500), fontsize=12)
plt.arrow(9, 2200, -5, 0, head_width=125, head_length=0.5, fc='k', ec='k', lw=.5)

plt.xlim(0.01, 1000)
plt.ylim(0, 3000)
plt.ylabel('Frequency')
plt.xlabel('Channel aspect ratio, $\\alpha$')
plt.legend(['Laboratory flumes', 'Natural channels'], loc=1, fancybox=True, framealpha=0)

sns.despine()
# plt.savefig(f'figures/figure_1_panel_a.pdf', bbox_inches='tight')
plt.show()

##### Figure 2

In [ ]:
fig, (ax0, ax1, ax2) = plt.subplots(nrows=1, ncols=3, sharex=False, sharey=False, figsize=(figwidth, figwidth/3), layout='constrained')
if global_cmap == 'cividis':
    cmaplines = mpl.cm.get_cmap('viridis')
elif global_cmap == 'viridis':
    cmaplines = mpl.cm.get_cmap('cividis')
    
# Panel a
vmin, vmax = 0.1, 10
for key, group in df.groupby('data_source'):
    if key == 'Song, 1994': continue
    if key == 'New experiments':
        ax0.scatter(group.tau8, group.q8, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], edgecolor='k', zorder=1)
    else:
        scatter0 = ax0.scatter(group.tau8, group.q8, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], zorder=0)

x = np.logspace(np.log10(0.0496),0.5,1000)
ax0.loglog(x, wp_func(x), 'k--', label='Best fit $\\alpha > 8$', lw=1.2)

# Panel b
vmin, vmax = 0.03, 3
for key, group in df.groupby('data_source'):
    if key == 'Song, 1994': ax1.scatter(group.alpha, group.q8, facecolors='none', edgecolor='gray', s=group.markersize, marker=group.m.values[0], label='{}'.format(key))
    elif key == 'New experiments':
        scatter1 = ax1.scatter(group.alpha, group.q8, c='k', s=group.markersize, marker=group.m.values[0], label='{}'.format(key), edgecolor='k', zorder=1)
    else:
        scatter1 = ax1.scatter(group.alpha, group.q8, c='k', s=group.markersize, marker=group.m.values[0], label='{}'.format(key))

# Panel c
vmin, vmax = -0.1, 1.1
for key, group in df.groupby('data_source'):
    if key == 'Williams, 1970': zorder = 1
    else: zorder = 2
    twt = calc_FVB(group.f, group.alpha, group.Re, output='twt')
    y = group.tau_m/group.tau
    if key == 'Song, 1994':
        ax2.scatter(group.alpha, y, facecolors='w', alpha=1, s=group.markersize/2, marker=group.m.values[0], edgecolor='k', zorder=zorder+1)
        scatter2 = ax2.scatter(group.alpha+1000, y, c=twt, cmap=cmaplines, norm=mcolors.Normalize(vmin=vmin, vmax=vmax), alpha=1, s=group.markersize/3, marker=group.m.values[0], zorder=zorder)
    elif key == 'New experiments':
        ax2.scatter(group.alpha, y, c='k', alpha=0.7, s=group.markersize/3, marker=group.m.values[0], edgecolor='k', zorder=3)
    else:
        ax2.scatter(group.alpha, y, c='k', alpha=0.7, s=group.markersize/3, marker=group.m.values[0], zorder=zorder)
    
# plot tauinf/tauinf
x = np.logspace(-2,2,100)
ax2.loglog(x, np.ones_like(x), 'k-', lw=1.2, zorder=4, label='Upper bound ($\\tau = \\tau_\infty$)')
# plot FRtau_inf/tauinf
delta_tau_FR = lambda x: 1/(1 + 2/x)
ax2.loglog(x, delta_tau_FR(x), 'k-.', lw=1.2, zorder=4, label='Lower bound ($\\tau = \\rho g RS$)')

# plot FWFRtau_inf/tauinf
norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
delta_tau = lambda x, twt: 1 - twt/(1+x/2)
twtvals = np.linspace(vmin, vmax, 25)
for twt in twtvals:
    ax2.loglog(x, delta_tau(x, twt), '-', lw=0.8, color=cmaplines(norm(twt)), zorder=0)  

# figure finishings
ax0.set_xlabel('Dimensionless bed stress, $\\tau^*_\infty$')
ax0.set_ylabel('Dimensionless sediment flux, $q^*$')
ax1.set_xlabel('Aspect ratio, $\\alpha$')
ax1.set_ylabel('Dimensionless sediment flux, $q^*$')
ax2.set_xlabel('Aspect ratio, $\\alpha$')
ax2.set_ylabel('Sidewall correction, $F_{q_s} = \\tau_{q_s}/\\tau_\infty$')

cax0 = ax0.inset_axes([0.5, 0.2, 0.5, 0.04])
fig.colorbar(scatter0, cax=cax0, orientation='horizontal', label='$\\alpha$')

# divider = make_axes_locatable(ax2)
# cax2 = divider.append_axes("right", size='3%', pad=0.05)
cax2 = ax2.inset_axes([0.5, 0.2, 0.5, 0.04])
cbar = fig.colorbar(scatter2, cax=cax2, orientation='horizontal', label='$\\tau_{wall}/\\tau$')

ax0.set_xlim(0.03, 3)
ax0.set_ylim(1e-4, 100)
ax1.set_xlim(0.05, 50)
ax1.set_ylim(1e-4, 100)
ax2.set_xlim(0.05, 50)
ax2.set_ylim(0.05, 3)

ax1.set_xscale('log'), ax1.set_yscale('log')

ax0.text(0.95, 0.99, 'a', fontsize=15, transform=ax0.transAxes)
# ax0.text(0.6, 0.05, 'Wong & Parker, 2006', fontsize=9, transform=ax0.transAxes)
ax1.text(0.95, 0.99, 'b', fontsize=15, transform=ax1.transAxes)
ax2.text(0.95, 0.99, 'c', fontsize=15, transform=ax2.transAxes)
# ax2.text(0.15, .94, '$F_{q_s}$ as $f(\\tau_{wall}/\\tau)$', fontsize=12, transform=ax2.transAxes)
# x = np.linspace(0.07,0.12,10)
# ax2.loglog(x, 1.7*np.ones_like(x), c=mpl.cm.get_cmap(global_cmap)(0.5), lw=0.8)

ax0.legend(frameon=False, loc=2)
ax1.legend(frameon=False, loc=2, ncols=2, fontsize=9)
ax1.legend_.set_bbox_to_anchor((0, 1.01))
ax2.legend(frameon=False, loc=2, ncols=1)
ax2.legend_.set_bbox_to_anchor((0, 1.03))

sns.despine()
# plt.savefig(f'figure_2.pdf', bbox_inches='tight')
plt.show()

##### Figure 3

In [ ]:
fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(nrows=2, ncols=2, sharex=True, sharey=True, figsize=(figwidth, figwidth/1.7), layout='constrained')
axs = [ax0, ax1, ax2, ax3]
vmin, vmax = 0.02, 2

def fvb_lim(alpha, Re):
    a, b = 0.018409368, 0.224072094
    return (9*a / (1 + alpha/2)) * ss.lambertw((3/a)*(1+alpha/2)*b**3*Re)**(-2)

def fg_lim(alpha, ro):
    beta = (3/4/kappa - 1/kappa/np.pi**2)**2
    phi = np.exp(-(alpha/np.pi)**(3/2))
    psi = beta + (2/alpha) * (np.log(ro*np.exp(-1))/kappa - (1 + phi)**3/12/kappa + ((1 + np.sqrt(phi))/2)*(1+np.sin(np.pi*phi)/(np.pi*(1+phi))))**2 
    return 1 / psi / (1 + alpha/2)

ro = 340
Re = 4e4

# panel a and b
for key, group in df.groupby('data_source'):
    if key == 'Song, 1994': ax0.scatter(group.alpha, group.f, facecolors='none', edgecolor='k', s=group.markersize*3, marker=group.m.values[0], alpha=1, zorder=20)
    if key == 'New experiments': 
        ax0.scatter(group.alpha, group.f, facecolors='none', edgecolor='k', s=group.markersize*3, marker=group.m.values[0], alpha=1, zorder=20)
    scatter0 = ax0.scatter(group.alpha, group.f, c=group.tau_m/group.tau, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize*3, marker=group.m.values[0], alpha=0.75)

    if key == 'New experiments':
        ax0.scatter(group.alpha+1000, group.f, facecolors='w', edgecolors='k', linewidths=1, s=group.markersize*3, marker=group.m.values[0], alpha=0.75, label='{}'.format(key))
        scatter0 = ax1.scatter(group.alpha, group.f, facecolors='w', edgecolors='k', linewidths=1, s=group.markersize*1.5, marker=group.m.values[0], alpha=0.75)
        scatter0 = ax2.scatter(group.alpha, group.f, facecolors='w', edgecolors='k', linewidths=1, s=group.markersize*1.5, marker=group.m.values[0], alpha=0.75)
        scatter0 = ax3.scatter(group.alpha, group.f, facecolors='w', edgecolors='k', linewidths=1, s=group.markersize*1.5, marker=group.m.values[0], alpha=0.75)
    elif key == 'Song, 1994':
        ax0.scatter(group.alpha+1000, group.f, facecolors='none', edgecolor='k', s=group.markersize*3, marker=group.m.values[0], alpha=0.75, label='{}'.format(key))
        scatter0 = ax1.scatter(group.alpha, group.f, facecolors='w', edgecolor='k', s=group.markersize, marker=group.m.values[0], zorder=20)
        scatter0 = ax2.scatter(group.alpha, group.f, facecolors='w', edgecolor='k', s=group.markersize, marker=group.m.values[0], zorder=20)
        scatter0 = ax3.scatter(group.alpha, group.f, facecolors='w', edgecolor='k', s=group.markersize, marker=group.m.values[0], zorder=20)
    else:
        ax0.scatter(group.alpha+1000, group.f, c='k', s=group.markersize*3, marker=group.m.values[0], alpha=0.75, label='{}'.format(key))
        scatter0 = ax1.scatter(group.alpha, group.f, c='k', s=group.markersize, marker=group.m.values[0], alpha=0.75)
        scatter0 = ax2.scatter(group.alpha, group.f, c='k', s=group.markersize, marker=group.m.values[0], alpha=0.75)
        scatter0 = ax3.scatter(group.alpha, group.f, c='k', s=group.markersize, marker=group.m.values[0], alpha=0.75)

    
# I calculate this for 10000 values in the publication (takes a while)
alphas = np.linspace(0.05,50,100) 
# alphas = np.linspace(0.05,50,10000) 
# I calculate this for 2500 values in the publication (takes a while)
fs = np.linspace(0.0008,0.02,25) 
# fs = np.linspace(0.0008,0.02,2500) 

extent = [alphas[0], alphas[-1], fs[0], fs[-1]]
FRs = np.zeros((len(fs), len(alphas)))
for i, f in enumerate(fs):
    for j, alpha in enumerate(alphas):
        FRs[i,j] = calc_FR(alpha) 
im1 = ax1.imshow(FRs, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), extent=extent, aspect='auto', origin='lower', cmap=global_cmap)

   
# panel c and d
FGs = np.zeros((len(fs), len(alphas)))
FVBs = np.zeros((len(fs), len(alphas)))
for i, f in enumerate(fs):
    for j, alpha in enumerate(alphas):
        FVBs[i,j] = calc_FVB(f, alpha, Re) 
        FGs[i,j] = calc_FG(f, alpha, ro) 
    
im2 = ax2.imshow(FVBs, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), extent=extent, aspect='auto', origin='lower', cmap=global_cmap)
im3 = ax3.imshow(FGs, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), extent=extent, aspect='auto', origin='lower', cmap=global_cmap)

# mask = np.where((FVBs/FGs < 1/1.2) | (FVBs/FGs > 1.2), 0, 1)
# mask = np.where(np.isfinite(FGs), mask, np.nan)
# ax0.contourf(mask, extent=extent, levels=[0.99, 1], colors='black', alpha=0.1, zorder=10)
# ax0.contour(mask, extent=extent, levels=[0.99], colors='black', linewidths=0.5, zorder=0)

ax2.plot(alphas, fvb_lim(alphas, Re), 'k--', lw=1)
ax3.plot(alphas, fg_lim(alphas, ro), 'k--', lw=1)
ax3.fill_between(alphas, np.ones_like(alphas)*extent[2], fg_lim(alphas, ro), color='none', hatch='...', edgecolor='k', lw=0.25, alpha=0.25)

# figure finishings
ax0.legend(ncols=2, frameon=False)

ax0.set_ylabel('Friction factor, $f$')
ax2.set_ylabel('Friction factor, $f$')

ax2.set_xlabel('Aspect ratio, $\\alpha$')
ax3.set_xlabel('Aspect ratio, $\\alpha$')

cbar = fig.colorbar(im2, ax=axs, label='Sidwall correction, $F$', pad=0.025)

bboxdict = dict(facecolor='white', edgecolor='none', pad=0.25, alpha=0.8, boxstyle='round')
ax0.text(0.97, 0.94, 'a', fontsize=15, transform=ax0.transAxes)
ax0.text(0.05, 0.9, '$F_{q_s} = \\tau_{q_s}/\\tau_\infty$ (flux-inferred)', fontsize=16, transform=ax0.transAxes, bbox=bboxdict)
ax1.text(0.97, 0.94, 'b', fontsize=15, transform=ax1.transAxes)
ax1.text(0.05, 0.9, '$F_R$ (hydraulic radius)', fontsize=16, transform=ax1.transAxes, bbox=bboxdict)
ax2.text(0.97, 0.94, 'c', fontsize=15, transform=ax2.transAxes)
ax2.text(0.05, 0.9, '$F_{VB}$ (Vanoni-Brooks)', fontsize=16, transform=ax2.transAxes, bbox=bboxdict)
ax2.text(0.95, 0.1, '$Re = 40,000$', ha='right', fontsize=16, transform=ax2.transAxes, bbox=bboxdict)
ax3.text(0.97, 0.94, 'd', fontsize=15, transform=ax3.transAxes)
ax3.text(0.05, 0.9, '$F_G$ (Guo)', fontsize=16, transform=ax3.transAxes, bbox=bboxdict)
ax3.text(0.95, 0.1, '$r_o = 340$', ha='right', fontsize=16, transform=ax3.transAxes, bbox=bboxdict)

ax0.set_xlim(extent[0], extent[1])
ax0.set_ylim(extent[2], extent[3])
ax0.set_xscale('log'), ax0.set_yscale('log')

sns.despine()
# plt.savefig(f'figure_3.pdf', bbox_inches='tight')
plt.show()

##### Figure 4

In [ ]:
fig, ((ax0, ax1), (ax2, ax3)) = plt.subplots(nrows=2, ncols=2, sharex='col', sharey=False, figsize=(figwidth, figwidth/1.7), layout='constrained')
axs = [ax0, ax1, ax2, ax3]
vmin, vmax = 0.1, 10

# panel a
for key, group in df.groupby('data_source'):
    if key == 'Deal et al., 2023':  
        ax2.scatter(group.tau_m, group.FG * group.tau, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], zorder=1)
        ax0.scatter(group.tau_m, group.FVB * group.tau, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], zorder=1)
        ax0.scatter(group.tau_m+1000, group.FVB * group.tau, c='k', s=group.markersize, marker=group.m.values[0], label='{}'.format(key), zorder=1)
    elif key == 'New experiments':
        ax2.scatter(group.tau_m, group.FG * group.tau, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], edgecolor='k', zorder=2)
        ax0.scatter(group.tau_m, group.FVB * group.tau, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], edgecolor='k', zorder=2)
        ax0.scatter(group.tau_m+1000, group.FVB * group.tau, c='k', s=group.markersize, marker=group.m.values[0], edgecolor='k', label='{}'.format(key), zorder=2)
    elif key == 'Song, 1994':
        ax2.scatter(group.tau_m, group.FG * group.tau, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], zorder=0)
        ax0.scatter(group.tau_m, group.FVB * group.tau, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], zorder=0)
        ax0.scatter(group.tau_m+1000, group.FVB * group.tau, facecolors='none', edgecolor='k', s=group.markersize, marker=group.m.values[0], label='{}'.format(key), zorder=0)
    else:
        scatter2 = ax2.scatter(group.tau_m, group.FG * group.tau, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], zorder=0)
        scatter0 = ax0.scatter(group.tau_m, group.FVB * group.tau, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], zorder=0)
        ax0.scatter(group.tau_m+1000, group.FVB * group.tau, c='k', s=group.markersize, marker=group.m.values[0], label='{}'.format(key), zorder=0)

    if key == 'Song, 1994':
        ax0.scatter(group.tau_m, group.FVB * group.tau, facecolors='none', s=group.markersize, marker=group.m.values[0], edgecolor='k', zorder=10)
        ax2.scatter(group.tau_m, group.FG * group.tau, facecolors='none', s=group.markersize, marker=group.m.values[0], edgecolor='k', zorder=10)
    
        
# plot 1:1, 2:1, 1:2 lines        
x = np.linspace(0, 100)
ax2.loglog(x, 2*x, 'k:')
ax0.loglog(x, 2*x, 'k:')
ax2.loglog(x, x/2, 'k:', label='2:1 or 1:2')
ax0.loglog(x, x/2, 'k:', label='2:1 or 1:2')
ax2.loglog(x, x, 'k--', label='1:1')
ax0.loglog(x, x, 'k--', label='1:1')

# panel b
for key, group in df.groupby('data_source'):
    tauG = group.FG * group.tau
    tauVB = group.FVB * group.tau
    if key == 'New experiments':
        ax3.scatter(group.alpha, group.tau_m/tauG, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], edgecolor='k', label='{}'.format(key), zorder=2)
        ax1.scatter(group.alpha, group.tau_m/tauVB, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], edgecolor='k', label='{}'.format(key), zorder=2)
    elif key == 'Song, 1994':
        ax3.scatter(group.alpha, group.f, facecolors='none', edgecolor='k', s=group.markersize, marker=group.m.values[0], alpha=1, zorder=0)
        ax3.scatter(group.alpha, group.tau_m/tauG, facecolors='none', c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], label='{}'.format(key), zorder=1)
        ax1.scatter(group.alpha, group.tau_m/tauVB, facecolors='none', c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], label='{}'.format(key), zorder=1)
    else:
        scatter3 = ax3.scatter(group.alpha, group.tau_m/tauG, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], label='{}'.format(key), zorder=0)
        scatter1 = ax1.scatter(group.alpha, group.tau_m/tauVB, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], label='{}'.format(key), zorder=0)

    if key == 'Song, 1994':
        ax3.scatter(group.alpha, group.tau_m/tauG, facecolors='none', s=group.markersize, marker=group.m.values[0], edgecolor='k', zorder=10)
        ax1.scatter(group.alpha, group.tau_m/tauVB, facecolors='none', s=group.markersize, marker=group.m.values[0], edgecolor='k', zorder=10)
    
# plot 1:1, 2:1, 1:2 lines
x = np.linspace(0, 50, 100)
ax3.plot(x, np.ones_like(x), 'k--', lw=0.8)
ax1.plot(x, np.ones_like(x), 'k--', lw=0.8)
ax3.plot(x, 2*np.ones_like(x), 'k:', lw=0.8)
ax1.plot(x, 2*np.ones_like(x), 'k:', lw=0.8)
ax3.plot(x, 0.5*np.ones_like(x), 'k:', lw=0.8)
ax1.plot(x, 0.5*np.ones_like(x), 'k:', lw=0.8)

# figure finishings
ax0.text(0.99, 0.93, 'a', fontsize=15, transform=ax0.transAxes)
ax1.text(0.99, 0.93, 'b', fontsize=15, transform=ax1.transAxes)
ax2.text(0.99, 0.93, 'c', fontsize=15, transform=ax2.transAxes)
ax3.text(0.99, 0.93, 'd', fontsize=15, transform=ax3.transAxes)

ax2.set_ylabel('Corrected stress, $F_G\\tau_{\infty}$ (Pa)')
ax3.set_ylabel('Relative error, $\\tau_{q_s}/(F_G\\tau_{\infty})$')
ax0.set_ylabel('Corrected stress, $F_{VB}\\tau_{\infty}$ (Pa)')
ax1.set_ylabel('Relative error, $\\tau_{q_s}/(F_{VB}\\tau_{\infty})$')
ax2.set_xlabel('Flux-inferred or measured stress (Pa)')
ax3.set_xlabel('Aspect ratio, $\\alpha$')

ax3.set_xscale('log')
ax3.set_yscale('log')
ax1.set_yscale('log')
ax1.set_yscale('log')

ax2.set_xlim(0.5, 100)
ax2.set_ylim(0.5, 100)
ax3.set_xlim(0.05, 30)
ax3.set_ylim(0.1, 10)
ax0.set_xlim(0.5, 100)
ax0.set_ylim(0.5, 100)
ax1.set_xlim(0.05, 30)
ax1.set_ylim(0.1, 10)

ax0.legend(frameon=False, loc=2)
ax2.text(0.95, .1, 'Guo', ha='right', fontsize=16, transform=ax2.transAxes)
ax3.text(0.95, .1, 'Guo', ha='right', fontsize=16, transform=ax3.transAxes)
ax0.text(0.95, .1, 'Vanoni-Brooks', ha='right', fontsize=16, transform=ax0.transAxes)
ax1.text(0.95, .1, 'Vanoni-Brooks', ha='right', fontsize=16, transform=ax1.transAxes)

# cbar = fig.colorbar(scatter0, ax=axs, label='Aspect ratio, $\\alpha$', pad=0.025)
cax2 = ax2.inset_axes([0.05, 0.9, 0.5, 0.04])
cbar = fig.colorbar(scatter2, cax=cax2, orientation='horizontal', label='Aspect ratio, $\\alpha$')

sns.despine()
# fig.savefig(f'figure_4.pdf', bbox_inches='tight')
plt.show()

##### Figure 5

In [ ]:
fig, (ax0, ax1) = plt.subplots(nrows=1, ncols=2, sharex=False, sharey=True, figsize=(figwidth, figwidth/3), layout='constrained')
axs = [ax0, ax1]

# panel a
for key, group in df.groupby('data_source'):
    if key == 'Song, 1994': continue
    if key == 'Deal et al., 2023':
        ax0.scatter(group.FVB * group.tau8+1000, group.q8, c='k', s=group.markersize, marker=group.m.values[0], label='{}'.format(key), zorder=1)
        ax0.scatter(group.FVB * group.tau8, group.q8, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], zorder=1)
    elif key == 'New experiments':
        ax0.scatter(group.FVB * group.tau8+1000, group.q8, c='k', s=group.markersize, marker=group.m.values[0], label='{}'.format(key), zorder=2, edgecolor='k')
        ax0.scatter(group.FVB * group.tau8, group.q8, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], zorder=2, edgecolor='k')
    else:
        scatter0 = ax0.scatter(group.FVB * group.tau8+1000, group.q8, c='k', s=group.markersize, marker=group.m.values[0], label='{}'.format(key), zorder=0)
        scatter0 = ax0.scatter(group.FVB * group.tau8, group.q8, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], zorder=0)
        
# plot Wong and Parker    
x = np.logspace(np.log10(0.0496),0.5,1000)
ax0.loglog(x, wp_func(x), 'k--')

# panel b
vmin, vmax = 0.1, 10
for key, group in df.groupby('data_source'):
    if key == 'Song, 1994': continue   
    if key == 'Deal et al., 2023':
        cf = group.FG / group.FR
        print(cf.mean(), cf.min(), cf.max(), cf.std())
        ax1.scatter(group.FG * group.tau8, group.q8, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], zorder=1)
    elif key == 'New experiments':
        ax1.scatter(group.FG * group.tau8, group.q8, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], zorder=2, edgecolor='k')
    else:
        scatter1 = ax1.scatter(group.FG * group.tau8, group.q8, c=group.alpha, cmap=global_cmap, norm=mcolors.LogNorm(vmin=vmin, vmax=vmax), s=group.markersize, marker=group.m.values[0], zorder=0)
  
# plot Wong and Parker    
ax1.loglog(x, wp_func(x), 'k--', label='Best fit $\\alpha > 8$')

# figure finishings
ax0.text(0.9, 0.95, 'a', fontsize=15, transform=ax0.transAxes)
ax0.text(0.1, 0.9, 'Vanoni-Brooks', fontsize=16, transform=ax0.transAxes)
ax1.text(0.9, 0.95, 'b', fontsize=15, transform=ax1.transAxes)
ax1.text(0.1, 0.9, 'Guo', fontsize=16, transform=ax1.transAxes)

ax0.set_xlabel('Corrected dimensionless bed stress, $F_{VB}\\tau_\infty^*$')
ax0.set_ylabel('Dimensionless sediment flux, $q^*$')
ax1.set_xlabel('Corrected dimensionless bed stress, $F_G\\tau_\infty^*$')

ax0.legend(frameon=False, loc=4)
ax1.legend(frameon=False, loc=4)
ax1.legend_.set_bbox_to_anchor((1, 0.25))

ax0.set_xlim(0.01, 1)
ax0.set_ylim(.5e-4, .5e2)
ax1.set_xlim(0.01, 1)
ax1.set_ylim(.5e-4, .5e2)

# cbar = fig.colorbar(scatter1, ax=axs, label='Aspect ratio, $\\alpha$')
cax1 = ax1.inset_axes([0.5, 0.2, 0.5, 0.04])
cbar = fig.colorbar(scatter1, cax=cax1, orientation='horizontal', label='Aspect ratio, $\\alpha$')

sns.despine()
# plt.savefig(f'figure_5.pdf', bbox_inches='tight')
plt.show()

##### Figure 6

In [ ]:
def fit_model_standard(tau8, q8):#, dtau8, dq8):
    def func(x, c, xc): return c*(x - xc)**(3/2)

    x_data = tau8
    # dx_data = dtau8
    y_data = q8
    # dy_data = dq8

    model = lm.Model(func)
    model.set_param_hint('c', value=6, min=0, max=100)
    model.set_param_hint('xc', value=0.1, min=0, max=np.nanmin(x_data))
    params = model.make_params()
    # dx_plot = dx_data.copy()
    # dy_plot = dy_data.copy()
    # x_plot = x_data.copy()
    # y_plot = y_data.copy()
    results = model.fit(y_data[np.isfinite(x_data)], params, x=x_data[np.isfinite(x_data)])#, weights=1/dy_data[np.isfinite(x_data)])
    xc = results.result.params['xc'].value
    dxc = results.result.params['xc'].stderr
    c = results.result.params['c'].value
    dc = results.result.params['c'].stderr
    rmsd_standard = np.sqrt(np.mean((np.log10(y_data) - np.log10(func(x_data, c, xc)))**2))
    
    return func, xc, c, dxc, dc, rmsd_standard

fig, ((ax0, ax1), (ax2, ax3), (ax4, ax5)) = plt.subplots(nrows=3, ncols=2, sharex=True, sharey=True, figsize=(figwidth, figwidth), layout='constrained')
fig.set_constrained_layout_pads(h_pad=0.25, w_pad=0.25)  # Adjust horizontal and vertical padding

ax2b = inset_axes(ax2, width="40%", height="40%", loc='upper left', borderpad=1)
ax4b = inset_axes(ax4, width="40%", height="40%", loc='upper left', borderpad=1)
# define special figure parameters
marker = {'sphere': 'o', 'gravel': '+'}
color = {'wide': '#0072B2', 'narrow': '#E69F00'}
ms = {'sphere': 5, 'gravel': 8}
labels = {'narrowsphere': 'spheres, narrow ($\\alpha \\approx 0.1$)', 'narrowgravel': 'gravel, narrow ($\\alpha \\approx 0.1$)', 'widesphere': 'spheres, wide ($\\alpha \\approx 1.2$)', 'widegravel': 'gravel, wide ($\\alpha \\approx 1.2$)'}
lwfunc = 0.8

for key, group in df.groupby('data_source'):
    if key == 'Deal et al., 2023' or 'New experiments':
        for gkey, grain in group.groupby('grain_kind'):
            if (gkey == 'sphere') or (gkey == 'gravel'):
                for fkey, flume in grain.groupby('flume_width'):
                    # panel a
                    ax0.plot(grain.FR * grain.tau8, grain.q8, '.', color=color[fkey], marker=marker[gkey], ms=ms[gkey], label=labels[fkey+gkey])
                    
                    # panel b
                    ax1.plot(grain.FS * grain.FR * grain.tau8, grain.q8, '.', color=color[fkey], marker=marker[gkey], ms=ms[gkey])
                    
                    if key == 'Deal et al., 2023': 
                        print(fkey, (grain.FG/grain.FR).mean(), (grain.FG/grain.FR).std())
                        a, s = 0.35, ms[gkey]
                    else: 
                        a, s = 1, ms[gkey]
                    # panel c
                    ax2.plot(grain.FVB * grain.tau8, grain.q8, '.', color=color[fkey], marker=marker[gkey], ms=s, alpha=a)

                    # panel d
                    ax3.plot(grain.FVB * grain.FS * grain.tau8, grain.q8, '.', color=color[fkey], marker=marker[gkey], ms=s, alpha=a)

                    # panel e
                    ax4.plot(grain.FG * grain.tau8, grain.q8, '.', color=color[fkey], marker=marker[gkey], ms=ms[gkey])
                    
                    # panel f
                    ax5.plot(grain.FG * grain.FS * grain.tau8, grain.q8, '.', color=color[fkey], marker=marker[gkey], ms=ms[gkey])
                    
                    # inset to panel c
                    # what we did in the nature paper
                    if fkey == 'narrow': 
                        if gkey == 'sphere':
                            ax2b.plot(2.41 * grain.FR * grain.tau8, grain.q8, '.', color=color[fkey], marker=marker[gkey], ms=ms[gkey], label='$F_C = 2.4F_R$')
                        else:
                            ax2b.plot(2.41 * grain.FR * grain.tau8, grain.q8, '.', color=color[fkey], marker=marker[gkey], ms=ms[gkey])
                            ax2b.plot(3.45e-2, 0.6, '.', color=color[fkey], marker=marker[gkey], ms=ms[gkey])
                    else: 
                        if gkey == 'sphere':
                            ax2b.plot(grain.FVB * grain.tau8, grain.q8, '.', color=color[fkey], marker=marker[gkey], ms=ms[gkey], label='$F_C = F_{VB}$')
                        else:
                            ax2b.plot(grain.FVB * grain.tau8, grain.q8, '.', color=color[fkey], marker=marker[gkey], ms=ms[gkey])
                            ax2b.plot(3.45e-2, 0.13, '.', color=color[fkey], marker=marker[gkey], ms=ms[gkey])
                            
                    # inset to panel e
                    # what we did in the nature paper
                    if fkey == 'narrow': 
                        if gkey == 'sphere':
                            ax4b.plot(2.41 * grain.FR * grain.tau8, grain.q8, '.', color=color[fkey], marker=marker[gkey], ms=ms[gkey], label='$F_C = 2.4F_R$')
                        else:
                            ax4b.plot(2.41 * grain.FR * grain.tau8, grain.q8, '.', color=color[fkey], marker=marker[gkey], ms=ms[gkey])
                            ax4b.plot(3.45e-2, 0.6, '.', color=color[fkey], marker=marker[gkey], ms=ms[gkey])
                    else: 
                        if gkey == 'sphere':
                            ax4b.plot(grain.FG * grain.tau8, grain.q8, '.', color=color[fkey], marker=marker[gkey], ms=ms[gkey], label='$F_C = F_G$')
                        else:
                            ax4b.plot(grain.FG * grain.tau8, grain.q8, '.', color=color[fkey], marker=marker[gkey], ms=ms[gkey])
                            ax4b.plot(3.45e-2, 0.13, '.', color=color[fkey], marker=marker[gkey], ms=ms[gkey])
                    
# grain = df[df.data_source.isin(['Deal et al., 2023'])]
grain = df[df.data_source.isin(['New experiments'])]
data_x, data_y = (grain.FVB * grain.FS * grain.tau8).values, grain.q8.values
# data_dx, data_dy = grain.dtau8.values, grain.dq8.values
func, xc, c, dxc, dc, rms = fit_model_standard(data_x, data_y)#, data_dx, data_dy)
print(c, dc, xc, dxc, rms)
x = np.logspace(np.log10(0.0496),0.5,1000)
# ax3.plot(x, func(x, c, xc), '-', c='k', lw=lwfunc, label='$q^* = {:.1f}(\\tau^*-{:.3f})^\\frac32$'.format(c, xc))
# ax5.plot(x, func(x, c, xc), '-', c='k', lw=lwfunc, label='$q^* = {:.1f}(\\tau^*-{:.3f})^\\frac32$'.format(c, xc))
                    
# plot Wong and Parker
x = np.logspace(np.log10(0.051),0.5,1000)
ax0.loglog(x, wp_func(x), 'k--', lw=lwfunc, label='Best fit $\\alpha > 8$')
ax1.loglog(x, wp_func(x), 'k--', lw=lwfunc)
ax2.loglog(x, wp_func(x), 'k--', lw=lwfunc)
ax3.loglog(x, wp_func(x), 'k--', lw=lwfunc)
ax4.loglog(x, wp_func(x), 'k--', lw=lwfunc)
ax5.loglog(x, wp_func(x), 'k--', lw=lwfunc)
ax2b.loglog(x, wp_func(x), 'k--', lw=lwfunc)
ax4b.loglog(x, wp_func(x), 'k--', lw=lwfunc)

ax0.set_xscale('log'), ax0.set_yscale('log')
ax0.set_ylim(1e-3, 2), ax0.set_xlim(2e-2, .5)
ax3.set_ylim(1e-3, 2), ax3.set_xlim(2e-2, .5)
ax2b.set_ylim(1e-3, 2), ax2b.set_xlim(2e-2, .5)
ax4b.set_ylim(1e-3, 2), ax4b.set_xlim(2e-2, .5)

ax0.set_xlabel(r'$F_R\tau^*_\infty$')
ax1.set_xlabel(r'$F_{shape}F_R\tau^*_\infty$')
ax2.set_xlabel(r'$F_{VB}\tau^*_\infty$')
ax3.set_xlabel(r'$F_{shape}F_{VB}\tau^*_\infty$')
ax4.set_xlabel(r'$F_G\tau^*_\infty$')
ax5.set_xlabel(r'$F_{shape}F_G\tau^*_\infty$')
ax2b.set_xlabel(r'$F_C\tau^*_\infty$', fontsize=11)
ax4b.set_xlabel(r'$F_C\tau^*_\infty$', fontsize=11)
ax0.set_ylabel(r'Dimensionless sediment flux, $q^*$')
ax2.set_ylabel(r'Dimensionless sediment flux, $q^*$')
ax4.set_ylabel(r'Dimensionless sediment flux, $q^*$')

ax0.legend(frameon=False, loc=4)
ax2.legend(frameon=False, loc=4)
ax3.legend(frameon=False, loc=4)
ax2b.legend(frameon=False, loc=2, fontsize=9)
ax2b.tick_params(labelleft=False)
ax2b.set_xscale('log'), ax2b.set_yscale('log')

ax4b.legend(frameon=False, loc=2, fontsize=9)
ax4b.tick_params(labelleft=False)
ax4b.set_xscale('log'), ax2b.set_yscale('log')

ax0.text(0.9, 0.925, 'a', fontsize=15, transform=ax0.transAxes)
ax1.text(0.9, 0.925, 'b', fontsize=15, transform=ax1.transAxes)
ax2.text(0.9, 0.925, 'c', fontsize=15, transform=ax2.transAxes)
ax3.text(0.9, 0.925, 'd', fontsize=15, transform=ax3.transAxes)
ax4.text(0.9, 0.925, 'e', fontsize=15, transform=ax4.transAxes)
ax5.text(0.9, 0.925, 'f', fontsize=15, transform=ax5.transAxes)

# Add a semitransparent box in figure coordinates
highlight_color = 'gray'  # Color of the box
highlight_alpha = 0.075    # Transparency level

# Coordinates for the box in figure coordinates (normalized 0 to 1)
x_locs = [[0.01, 1.025], [0.01, 1.025], [0.01, 1.025], [0.025, 0.5], [0.56, 1.01]] 
y_locs = [[0.035, 0.315], [0.38, 0.64], [0.71, .97], [0.02, 0.99], [0.02, .99]] 

# Labels for each rectangle
labels = [ "Guo Correction", "Vanoni-Brooks correction", "Hydraulic radius correction", 'Without shape correction', 'With shape correction']

# Add the rectangle to the figure
for i in range(5):
    x_start, x_end = x_locs[i]
    y_start, y_end = y_locs[i]
    if i < 3:
        rect = FancyBboxPatch(
                        (x_start, y_start),           # (x, y) lower left corner
                        x_end - x_start, y_end - y_start,             # width, height
                        boxstyle="round,pad=0.02",  # Rounded corners
                        linewidth=1,          # Border width
                        edgecolor='black',    # Border color
                        facecolor='none',     # Fill color
                        transform=fig.transFigure,
                        alpha=1             # Transparency (0=transparent, 1=opaque)
                    )
        fig.add_artist(rect)

    elif i > 2:
        # rect = Rectangle((x_start, y_start), x_end - x_start, y_end - y_start,
                        #  transform=fig.transFigure, facecolor=None, edgecolor='k', alpha=1, zorder=2)
        rect = FancyBboxPatch(
                        (x_start, y_start),           # (x, y) lower left corner
                        x_end - x_start, y_end - y_start,             # width, height
                        boxstyle="round,pad=0.02",  # Rounded corners
                        linewidth=1,          # Border width
                        edgecolor='none',    # Border color
                        facecolor='gray',     # Fill color
                        transform=fig.transFigure,
                        alpha=.1# Transparency (0=transparent, 1=opaque)
                    )
        fig.add_artist(rect)


    # Add vertical text at the right edge of the rectangle
    if i < 3: 
        fig.text(
            x_end + 0.03,  # Slightly inside the right edge of the rectangle
            (y_start + y_end) / 2,  # Centered vertically within the rectangle
            labels[i],  # Text label
            fontsize=16,  # Font size
            rotation=90,  # Rotate text vertically
            va='center',  # Vertical alignment
            ha='left',   # Horizontal alignment
        )
    # Add horiztonal text at the top edge of the rectangle
    else:
        fig.text(
            (x_start + x_end) / 2,  # Centered horizontally within the rectangle
            y_end + 0.025,  # Slightly inside the top edge of the rectangle
            labels[i],  # Text label
            fontsize=16,  # Font size
            va='bottom',  # Vertical alignment
            ha='center',   # Horizontal alignment
        )
    
sns.despine()
# plt.tight_layout()
# plt.savefig(f'figure_6.pdf', bbox_inches='tight')
plt.show()